In [1]:
import torch
import numpy as np
from ax import ChoiceParameterConfig
from ax.core import RangeParameter
from sklearn.metrics import precision_score, recall_score, f1_score
from core import *
from utils import *
from lark import Tree, Token
from pm4py import save_vis_petri_net
import pandas as pd

# SETTINGS
NARY = 1
PROBABILITIES = 0.2,0.2,0.2,0.4
FILE_PATH_PNG = "petri_net_output.png"
TRACE_ENC_REG = "data/regions_full.csv"
TRACE_ENC_TAS = "data/tasks_full.csv"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
learning_rate = 3e-4

In [2]:
'''def get_batch(split, train_data_X, train_data_Y, val_data_X, val_data_Y, batch_size, device):
    # Seleziona i dati corretti
    X_data = train_data_X if split == 'train' else val_data_X
    Y_data = train_data_Y if split == 'train' else val_data_Y

    # Genera indici casuali
    ix = torch.randint(len(X_data), (batch_size,))

    # Estrae e sposta sul device
    x = X_data[ix].to(device)
    y = Y_data[ix].to(device)

    return x, y'''

def get_batch(split, train_data, val_data, batch_size, block_size, device):
    # Seleziona i dati corretti
    data = train_data if split == 'train' else val_data

    # Genera indici casuali
    ix = torch.randint(len(data) - block_size, (batch_size,))

    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    x,y = x.to(device), y.to(device)
    return x, y


@torch.no_grad()
def estimate_loss(model, eval_iters, train_data, val_data, batch_size, block_size, device):
    out = {}
    model.eval()

    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            # Richiama get_batch passando i parametri ricevuti
            X, Y = get_batch(split, train_data, val_data, batch_size, block_size, device)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()

    model.train()
    return out

In [3]:
iterations = 15  # Quante task diverse
current_string = SEED_STRING
for _ in range(iterations):
    current_string = replace_random_underscore(current_string, PROBABILITIES)

process = replace_underscores(current_string)
tree = PARSER.parse(process)

if NARY:
    tree = createNAryTree(tree)

net = PetriNetP(tree)

save_vis_petri_net(
    net.net,
    net.initial_marking,
    net.final_marking,
    FILE_PATH_PNG,
    format="png"  # Specifica il formato
)

tree = Tree('loop', [Tree('parallel', [Tree('sequential', [Tree('loop', [Tree('parallel', [Tree('task', [Token('NAME', 'T1')]), Tree('task', [Token('NAME', 'T2')])])]), Tree('loop', [Tree('parallel', [Tree('loop', [Tree('task', [Token('NAME', 'T3')])]), Tree('task', [Token('NAME', 'T4')])])])]), Tree('loop', [Tree('sequential', [Tree('task', [Token('NAME', 'T5')]), Tree('loop', [Tree('task', [Token('NAME', 'T6')])])])])])])

# Oggetto Generator
generator = Generator(200000, net)

In [4]:
# Creazione matrice identità delle regioni
df_region_identity = pd.DataFrame.from_dict(net.node_identity, orient='index').sort_index()
df_region_identity.columns = ['X', '+', '->', '<>']
print(df_region_identity)

# Creazione matrice regioni-figli per le regioni
df_region_children = pd.Series(net.node_children).explode()
df_region_children = pd.crosstab(df_region_children.index, df_region_children)
df_region_children = df_region_children.reindex(index=net.regions, columns=net.regions + net.tasks, fill_value=0)
df_region_children = df_region_children.astype(int)
df_region_children.index.name = None
df_region_children.columns.name = None
print(df_region_children)

traceEncoded_regions, traceEncoded_tasks = getEncoding(generator.generatedTraces, net.regions, net.tasks, net.open_clauses, net.end_clauses)

num_regions = len([i for i in traceEncoded_regions.index if str(i).startswith('R')])
num_tasks = len([i for i in traceEncoded_tasks.index if str(i).startswith('T')])

traceEncoded_regions.to_csv(TRACE_ENC_REG, index=True)
traceEncoded_tasks.to_csv(TRACE_ENC_TAS, index=True)

df_traces = pd.concat([traceEncoded_regions, traceEncoded_tasks], axis=0)
print(df_traces)

df_traces = df_traces.T

df_tracescopy = df_traces.copy()

unique_columns = df_traces.drop_duplicates()
unique_tuple = [tuple(x) for x in unique_columns.values]

# 2. Creiamo i dizionari di mappatura
# bit_to_id: trasforma la colonna di 12 bit in un numero
# id_to_bit: trasforma il numero nei 12 bit originali (per la generazione)
bit_to_id = {v: i for i, v in enumerate(unique_tuple)}
id_to_bit = {i: v for i, v in enumerate(unique_tuple)}

vocab_size = len(unique_columns)

encode = lambda a: [bit_to_id[tuple(x)] for x in a]
decode = lambda b: [id_to_bit[x] for x in b]

data = torch.tensor(encode(df_traces.values), dtype=torch.long)

n = int(0.8 * len(df_traces))

train_data = data[:n]
val_data = data[n:]

     X  +  ->  <>
R0   1  0   0   0
R1   0  1   0   0
R10  0  0   0   1
R11  1  0   0   0
R12  0  1   0   0
R2   0  0   0   1
R3   1  0   0   0
R4   0  0   1   0
R5   0  0   0   1
R6   0  1   0   0
R7   0  0   0   1
R8   0  0   1   0
R9   0  1   0   0
     R0  R1  R10  R11  R12  R2  R3  R4  R5  R6  ...  T11  T12  T2  T3  T4  T5  \
R0    0   1    0    0    0   0   0   0   0   0  ...    0    0   0   0   0   0   
R1    0   0    0    0    0   1   0   1   1   0  ...    0    0   0   0   0   0   
R10   0   0    0    1    0   0   0   0   0   0  ...    0    0   0   0   0   0   
R11   0   0    0    0    1   0   0   0   0   0  ...    0    0   0   0   0   0   
R12   0   0    0    0    0   0   0   0   0   0  ...    1    1   0   0   0   0   
R2    0   0    0    0    0   0   1   0   0   0  ...    0    0   0   0   0   0   
R3    0   0    0    0    0   0   0   0   0   0  ...    0    0   1   1   0   0   
R4    0   0    0    0    0   0   0   0   0   0  ...    0    0   0   0   1   1   
R5    0   0    0   

In [5]:
def algo(n_head, n_layer, block_size, n_embd, dropout, learning_rate):
    batch_size = 32

    # Inizializzo modello
    model = BPMNTransformer(
        vocab_size,
        num_regions+num_tasks,
        block_size,
        n_embd,
        dropout,
        n_head,
        n_layer,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

    mini_iters = 300
    eval_iters = 200
    eval_interval = 250
    max_iters = 2000

    model.train() # Lo metto in modalità train

    for step in range(mini_iters):
        xb, yb = get_batch('train', train_data, val_data, batch_size, block_size, device)
        logits, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    model.eval()
    losses = estimate_loss(model, eval_iters, train_data, val_data, batch_size, block_size, device)

    print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    return losses['val']

In [6]:
import numpy as np
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig

client = Client()

parameters = [
    ChoiceParameterConfig(
        name = "n_head", parameter_type="int", values=[2,4,8], is_ordered=True
    ),
    RangeParameterConfig(
        name = "n_layer", parameter_type="int", bounds=(3,6)
    ),
    ChoiceParameterConfig(
        name = "block_size", parameter_type="int", values=[128,256], is_ordered=True
    ),
    ChoiceParameterConfig(
        name = "n_embd", parameter_type="int", values=[128,256,512], is_ordered=True
    ),
    RangeParameterConfig(
        name = "dropout", parameter_type="float", bounds=(0.25,0.45)
    ),
    RangeParameterConfig(
        name = "learning_rate", parameter_type="float", bounds=(1e-5, 1e-3)
    ),
]

client.configure_experiment(parameters=parameters)

metric_name = "loss"
objective = f"-{metric_name}"

client.configure_optimization(objective=objective)

In [7]:
for _ in range(10):
    trials = client.get_next_trials(max_trials=2)
    print("ciao")

    for trial_index, parameters in trials.items():
        n_head = parameters["n_head"]
        n_layer = parameters["n_layer"]
        block_size = parameters["block_size"]
        n_embd = parameters["n_embd"]
        dropout = parameters["dropout"]
        learning_rate = parameters["learning_rate"]

        result = algo(n_head, n_layer, block_size, n_embd, dropout, learning_rate)

        # Set raw_data as a dictionary with metric names as keys and results as values
        raw_data = {metric_name: (float(result), 1e-6)}

        # Complete the trial with the result
        client.complete_trial(trial_index=trial_index, raw_data=raw_data)

[INFO 04-15 15:57:07] ax.api.client: GenerationStrategy(name='Center+Sobol+MBM:fast', nodes=[CenterGenerationNode(next_node_name='Sobol'), GenerationNode(name='Sobol', generator_specs=[GeneratorSpec(generator_enum=Sobol, generator_key_override=None)], transition_criteria=[MinTrials(transition_to='MBM'), MinTrials(transition_to='MBM')], suggested_experiment_status=ExperimentStatus.INITIALIZATION, pausing_criteria=[MaxTrialsAwaitingData(threshold=5)]), GenerationNode(name='MBM', generator_specs=[GeneratorSpec(generator_enum=BoTorch, generator_key_override=None)], transition_criteria=None, suggested_experiment_status=ExperimentStatus.OPTIMIZATION, pausing_criteria=None)]) chosen based on user input and problem structure.
[INFO 04-15 15:57:07] ax.api.client: Generated new trial 0 with parameters {'n_head': 4, 'n_layer': 4, 'block_size': 256, 'n_embd': 256, 'dropout': 0.35, 'learning_rate': 0.000505} using GenerationNode CenterOfSearchSpace.
[INFO 04-15 15:57:07] ax.api.client: Generated ne

ciao


[INFO 04-15 15:58:31] ax.api.client: Trial 0 marked COMPLETED.


step <built-in function iter>: train loss 0.7081, val loss 0.7077


[INFO 04-15 15:58:56] ax.api.client: Trial 1 marked COMPLETED.
[INFO 04-15 15:58:56] ax.api.client: Generated new trial 2 with parameters {'n_head': 8, 'n_layer': 4, 'block_size': 256, 'n_embd': 512, 'dropout': 0.41841, 'learning_rate': 0.000629} using GenerationNode Sobol.
[INFO 04-15 15:58:56] ax.api.client: Generated new trial 3 with parameters {'n_head': 8, 'n_layer': 6, 'block_size': 128, 'n_embd': 256, 'dropout': 0.370527, 'learning_rate': 0.000833} using GenerationNode Sobol.


...

ciao


[INFO 04-15 16:29:03] ax.api.client: Trial 16 marked COMPLETED.


step <built-in function iter>: train loss 0.7173, val loss 0.7163


[INFO 04-15 16:29:47] ax.api.client: Trial 17 marked COMPLETED.


step <built-in function iter>: train loss 0.7793, val loss 0.7781


In [8]:
best_parameters, prediction, index, name = client.get_best_parameterization()
print("Best Parameters:", best_parameters)
print("Prediction (mean, variance):", prediction)

C:\Users\nicoz\PycharmProjects\ISI-MasterDegreeThesis\.venv\Lib\site-packages\gpytorch\likelihoods\noise_models.py:150: NumericalWarning: Very small noise values detected. This will likely lead to numerical instabilities. Rounding small noise values up to 1e-06.
  warnings.warn(
C:\Users\nicoz\PycharmProjects\ISI-MasterDegreeThesis\.venv\Lib\site-packages\gpytorch\likelihoods\noise_models.py:150: NumericalWarning: Very small noise values detected. This will likely lead to numerical instabilities. Rounding small noise values up to 1e-06.
  warnings.warn(
C:\Users\nicoz\PycharmProjects\ISI-MasterDegreeThesis\.venv\Lib\site-packages\gpytorch\likelihoods\noise_models.py:150: NumericalWarning: Very small noise values detected. This will likely lead to numerical instabilities. Rounding small noise values up to 1e-06.
  warnings.warn(
C:\Users\nicoz\PycharmProjects\ISI-MasterDegreeThesis\.venv\Lib\site-packages\gpytorch\likelihoods\noise_models.py:150: NumericalWarning: Very small noise value

Best Parameters: {'n_head': 4, 'n_layer': 5, 'block_size': 256, 'n_embd': 256, 'dropout': 0.25, 'learning_rate': 0.0008652270052434077}
Prediction (mean, variance): {'loss': (np.float64(0.6738083643383139), np.float64(1.270554147317205e-08))}


In [15]:
print(tree)

Tree('loop', [Tree('parallel', [Tree('sequential', [Tree('loop', [Tree('parallel', [Tree('task', [Token('NAME', 'T1')]), Tree('task', [Token('NAME', 'T2')])])]), Tree('loop', [Tree('parallel', [Tree('loop', [Tree('task', [Token('NAME', 'T3')])]), Tree('task', [Token('NAME', 'T4')])])])]), Tree('loop', [Tree('sequential', [Tree('task', [Token('NAME', 'T5')]), Tree('loop', [Tree('task', [Token('NAME', 'T6')])])])])])])
